In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os
import copy
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class RegressionModel(nn.Module):
    def __init__(self, layers_list):
        super().__init__()
        layers = []

        for i in range(len(layers_list) - 1):
            layers.append(nn.Linear(layers_list[i], layers_list[i + 1]))

            if i < len(layers_list) - 2:
                layers.append(nn.BatchNorm1d(layers_list[i + 1]))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(0.1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def get_loader(X, y, bs):
    dataset = TensorDataset(X, y)

    return DataLoader(
        dataset,
        batch_size=bs,
        shuffle=True,
        num_workers=30,
        pin_memory=True,
        persistent_workers=True
    )

def calc_metrics(obs, pred):
    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae = mean_absolute_error(obs, pred)
    r2 = r2_score(obs, pred)

    return rmse, mae, r2

def main():

    print('\n' + '=' * 60)
    print('🚀 5-Fold Cross Validation (Grouped by Day)')
    print('=' * 60)

    data_path = '~/private_data/20260918/CMAQ_trainingset_2015-2019_ALL.csv'
    data = pd.read_csv(data_path)

    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna()

    data['O3_bias'] = data['O3_true'] - data['O3']
    data['PM25_bias'] = data['PM25_true'] - data['PM25']

    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

    data['time'] = pd.to_datetime(data['time'])
    data['date'] = data['time'].dt.date

    doy = data['time'].dt.dayofyear

    data['doy_sin'] = np.sin(2 * np.pi * doy / 365)
    data['doy_cos'] = np.cos(2 * np.pi * doy / 365)

    data['POA_EC_ratio'] = data['POA'] / data['EC']

    exclude_cols = [
        'time',
        'date',
        'O3_true',
        'PM25_true',
        'O3_bias',
        'PM25_bias'
    ]

    feature_cols = [c for c in data.columns if c not in exclude_cols]

    print('\nFeatures:')
    print(feature_cols)

    input_dim = len(feature_cols)

    param_grid = {
        'hidden_configs': [
            [input_dim, 2048, 1024, 512, 256, 2],
            [input_dim, 1024, 512, 256, 128, 2],
            [input_dim, 512, 256, 128, 64, 2],
        ],
        'lrs': [0.01, 0.001],
        'batch_sizes': [512, 1024, 2048]
    }

    unique_dates = data['date'].unique()

    kf = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    print('\n📦 Preparing fold cache and recording splits...')

    fold_cache = {}
    
    # 【新增】用于记录每天在每一个 Fold 中扮演的角色
    split_records = []

    for fold_idx, (train_date_idx, test_date_idx) in enumerate(kf.split(unique_dates)):
        
        outer_train_dates = unique_dates[train_date_idx]
        test_dates = unique_dates[test_date_idx]

        train_dates, val_dates = train_test_split(
            outer_train_dates,
            test_size=0.2,
            random_state=42
        )

        # 【新增】将这一折的划分情况记录下来
        # 记录格式：日期、当前是第几个Fold、在该Fold中属于什么集合
        for d in train_dates:
            split_records.append({'date': d, 'fold': fold_idx + 1, 'set_type': 'train'})
        for d in val_dates:
            split_records.append({'date': d, 'fold': fold_idx + 1, 'set_type': 'val'})
        for d in test_dates:
            split_records.append({'date': d, 'fold': fold_idx + 1, 'set_type': 'test'})

        train_df = data[data['date'].isin(train_dates)].copy()
        val_df = data[data['date'].isin(val_dates)].copy()
        test_df = data[data['date'].isin(test_dates)].copy()

        scaler = StandardScaler()

        X_train = scaler.fit_transform(train_df[feature_cols])
        X_val = scaler.transform(val_df[feature_cols])
        X_test = scaler.transform(test_df[feature_cols])

        y_train = train_df[['O3_bias', 'PM25_bias']].values
        y_val = val_df[['O3_bias', 'PM25_bias']].values
        y_test = test_df[['O3_bias', 'PM25_bias']].values

        fold_cache[fold_idx] = {
            'X_train': torch.tensor(X_train.astype(np.float32)),
            'X_val': torch.tensor(X_val.astype(np.float32)).to(device),
            'X_test': torch.tensor(X_test.astype(np.float32)).to(device),

            'y_train': torch.tensor(y_train.astype(np.float32)),
            'y_val': torch.tensor(y_val.astype(np.float32)).to(device),
            'y_test': torch.tensor(y_test.astype(np.float32)).to(device),

            'test_df': test_df
        }

    # 【新增】将划分记录转换成 DataFrame
    split_df = pd.DataFrame(split_records)

    print('✅ Fold cache prepared!')

    all_results = []
    metrics_all = []

    for fold_idx in range(5):

        print('\n' + '=' * 60)
        print(f'🧪 Testing Fold: {fold_idx + 1}')
        print('=' * 60)

        cache = fold_cache[fold_idx]

        X_train = cache['X_train']
        X_val = cache['X_val']
        X_test = cache['X_test']

        y_train = cache['y_train']
        y_val = cache['y_val']
        y_test = cache['y_test']

        test_df = cache['test_df']

        print('\n🔎 Hyperparameter searching...')

        configs = [
            (h, lr, bs)
            for h in param_grid['hidden_configs']
            for lr in param_grid['lrs']
            for bs in param_grid['batch_sizes']
        ]

        best_cfg = None
        best_val_loss = float('inf')

        criterion = nn.MSELoss()

        for h_cfg, lr, bs in configs:

            print(f'Testing h={h_cfg}, lr={lr}, bs={bs}')

            train_loader = get_loader(X_train, y_train, bs)

            temp_model = RegressionModel(h_cfg).to(device)

            optimizer = torch.optim.Adam(
                temp_model.parameters(),
                lr=lr
            )

            for epoch in range(20):
                temp_model.train()
                for xb, yb in train_loader:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)
                    optimizer.zero_grad()
                    pred = temp_model(xb)
                    loss = criterion(pred, yb)
                    loss.backward()
                    optimizer.step()
            temp_model.eval()
            with torch.no_grad():
                val_loss = criterion(
                    temp_model(X_val),
                    y_val
                ).item()
            print(f'Val Loss: {val_loss:.6f}')
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_cfg = {
                    'h': h_cfg,
                    'lr': lr,
                    'bs': bs
                }
            del temp_model
            del optimizer

            torch.cuda.empty_cache()

        print('\n✅ Best Config:')
        print(best_cfg)

        train_loader = get_loader(X_train, y_train, best_cfg['bs'])

        model = RegressionModel(best_cfg['h']).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=best_cfg['lr']
        )

        criterion = nn.MSELoss()

        best_wts = None
        min_val_loss = float('inf')
        counter = 0

        print('\n🚀 Final Training...')

        for epoch in range(300):

            model.train()

            for xb, yb in train_loader:

                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                optimizer.zero_grad()

                pred = model(xb)

                loss = criterion(pred, yb)

                loss.backward()

                optimizer.step()

            model.eval()

            with torch.no_grad():

                val_loss = criterion(
                    model(X_val),
                    y_val
                ).item()

            if val_loss < min_val_loss:

                min_val_loss = val_loss
                best_wts = copy.deepcopy(model.state_dict())
                counter = 0

            else:

                counter += 1

            if (epoch + 1) % 20 == 0:
                print(f'Epoch [{epoch+1}/300] | Val Loss: {val_loss:.6f}')

            if counter >= 30:
                print(f'\n⏹ Early stop at epoch {epoch+1}')
                break

        model.load_state_dict(best_wts)

        model.eval()

        with torch.no_grad():
            pred_test = model(X_test).cpu().numpy()

        result_df = pd.DataFrame({

            'time': test_df['time'].values,
            'fold': fold_idx + 1,

            'O3_obs_bias': y_test.cpu().numpy()[:, 0],
            'O3_pred_bias': pred_test[:, 0],
            'O3_raw': test_df['O3'].values,
            'O3_obs': test_df['O3_true'].values,
            'O3_corrected': test_df['O3'].values + pred_test[:, 0],

            'PM25_obs_bias': y_test.cpu().numpy()[:, 1],
            'PM25_pred_bias': pred_test[:, 1],
            'PM25_raw': test_df['PM25'].values,
            'PM25_obs': test_df['PM25_true'].values,
            'PM25_corrected': test_df['PM25'].values + pred_test[:, 1]
        })

        o3_rmse, o3_mae, o3_r2 = calc_metrics(
            result_df['O3_obs_bias'],
            result_df['O3_pred_bias']
        )

        pm_rmse, pm_mae, pm_r2 = calc_metrics(
            result_df['PM25_obs_bias'],
            result_df['PM25_pred_bias']
        )

        metric_dict = {
            'fold': fold_idx + 1,
            'O3_RMSE': o3_rmse,
            'O3_MAE': o3_mae,
            'O3_R2': o3_r2,
            'PM25_RMSE': pm_rmse,
            'PM25_MAE': pm_mae,
            'PM25_R2': pm_r2
        }

        metrics_all.append(metric_dict)

        print('\n📊 Metrics')

        print(f'O3   | RMSE={o3_rmse:.3f} MAE={o3_mae:.3f} R2={o3_r2:.3f}')
        print(f'PM25 | RMSE={pm_rmse:.3f} MAE={pm_mae:.3f} R2={pm_r2:.3f}')

        all_results.append(result_df)

        del model
        del optimizer

        torch.cuda.empty_cache()

    final_df = pd.concat(all_results, ignore_index=True)
    metric_df = pd.DataFrame(metrics_all)
    o3_rmse, o3_mae, o3_r2 = calc_metrics(final_df['O3_obs_bias'], final_df['O3_pred_bias'])
    pm_rmse, pm_mae, pm_r2 = calc_metrics(final_df['PM25_obs_bias'], final_df['PM25_pred_bias'])
    print('\n' + '=' * 60)
    print('📌 OVERALL PERFORMANCE')
    print('=' * 60)

    print(f'O3   | RMSE={o3_rmse:.3f} MAE={o3_mae:.3f} R2={o3_r2:.3f}')
    print(f'PM25 | RMSE={pm_rmse:.3f} MAE={pm_mae:.3f} R2={pm_r2:.3f}')

    save_dir = 'CMAQ_5Fold_results'

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    final_df.to_csv(
        f'{save_dir}/5Fold_all_results.csv',
        index=False
    )
    metric_df.to_csv(f'{save_dir}/5Fold_metrics.csv', index=False)
    
    # 【新增】将记录划分情况的 CSV 写入对应的保存目录
    split_df.to_csv(f'{save_dir}/5Fold_date_splits.csv', index=False)

    overall_df = pd.DataFrame({
        'Species': ['O3', 'PM25'],
        'RMSE': [o3_rmse, pm_rmse],
        'MAE': [o3_mae, pm_mae],
        'R2': [o3_r2, pm_r2]
    })
    overall_df.to_csv(
        f'{save_dir}/5Fold_overall_metrics.csv',
        index=False
    )
    print('\n💾 Results saved:')
    print(save_dir)
    print('\n✅ Finished!')

if __name__ == '__main__':
    main()